In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from datetime import datetime
import os
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path








In [21]:

class DataParser:
    """
    class for reading in data from the Sri Lanka data
    """

    def __init__(
            self, 
            survey_id: str, 
            path_to_datasets: str, 
            path_to_targets: str = None,
            path_to_hh_info: str = Path("../../data/processed/hh_info.csv"),
            fsq24: bool = False
            ) -> None:
        
        self.survey_id = survey_id
        self.path_to_datasets = Path(path_to_datasets) 
        self.path_to_targets = Path(path_to_targets) if path_to_targets else None
        self.hh_info = pd.read_csv(path_to_hh_info)
        self.fsq24 = fsq24
        self.hhid = 'hhid'
        self.df = pd.DataFrame() 
        self.targets = pd.DataFrame()



    # read in the dataset for the features
    def read_datasets(self, dataset_name: str) -> pd.DataFrame:
        if self.fsq24:
            csv_path = self.path_to_datasets / "food_security_survey_2024" / "data-fs-sp_final-v2.xlsx"
            df = pd.read_excel(csv_path)
            self.df = df
        else:
            csv_path = self.path_to_datasets / "HIES_2019" / "HIES_2019" / f"{dataset_name}.csv"
            df = pd.read_csv(csv_path)
            self.df = df
        return df
    

    # read in the target variables
    
    def read_targets(self) -> pd.DataFrame:
        if self.path_to_targets is None:
            raise ValueError("No path_to_targets was provided during initialization.")

        self.targets = pd.read_csv(self.path_to_targets)
        self.targets = self.targets.merge(self.hh_info[["hhid",'adm2', 'month']])
        return self.targets
    
    # read in the shapefiles 

    # aggregate targets 
    def aggregate_target(self, groups:list = None, median: bool = True):
        """
        This method will aggregate the 
        """
        if groups is None: return self.targets
        else:
            self.targets = self.targets.groupby(groups).agg({
                'folate_mcg': 'median',
                'fe_mg': 'median',
                'zn_mg': 'median',
                'vita_rae_mcg':'median',
                'overall_mar': 'median'
            }).reset_index()
            return self.targets
    
   
    @staticmethod
    def make_hhid(df: pd.DataFrame)-> pd.DataFrame:
    # Ensure all parts are strings and handle missing values
        df['hhid'] = (
            df['psu'].astype(str).str.strip() +
            df['snumber'].astype(str).str.strip() +
            df['hhno'].astype(str).str.strip()
        ).astype(int)
        return df
    
     # make a plot, if fsq, only histogram or time-series
    
    def plot(self, x, y = None,x_lab:str, y_lab:str, title:str ):
        pass
    

    
class Income(DataParser):
    """
    Should this be an inherited class? only create the fucntions specific to the income variables?
    """

    def get_asw_perexp(self):


            if(self.fsq24):
                self.df['TotalConsumption'] = (self.df['TotalFexpMonthly'] + self.df['totalNFMonthly'])/self.df['HHSize']

                self.df['asw_perexp_poor'] = (
                self.df['TotalConsumption']<16000
                                    ).astype(int)

                self.df = self.df[['@_id', 'asw_perexp_poor']]
                return self.df
            else:
                self.make_hhid(self.df)
                self.df['asw_perexp_poor'] =(self.df['hhexppm']/self.df['hhsize']  < 8104).astype(int)

                self.df = self.df[['hhid','asw_perexp_poor']]
            return self.df
    




In [22]:
# set the paths
path_to_datasets = r"C:/Users/gabriel.battcock/OneDrive - World Food Programme/General - MIMI Project/Countries/Sri Lanka/data/"
path_to_targets = Path("../../data/processed/sl_ml_targets_2025-11-13.csv")



In [23]:
# create instances of the class to test
hies_19 = DataParser('HIES19', path_to_datasets,path_to_targets, fsq24 = False)

hies_19.read_datasets(dataset_name="HH_expenditure_hh_Income")
hies_19.df
hies_19.read_targets()
hies_19.aggregate_target()
hies_19.get_asw_perexp()


,hhid,asw_perexp_poor
0,11108811,0
1,11108831,0
2,11108851,0
3,11108861,0
4,11108871,0
...,...,...
19906,19208561,0
19907,19208571,1
19908,19208581,1
19909,19208591,0
